In [8]:
import os
from typing import TypedDict, Annotated, Literal, List, Optional
from dotenv import load_dotenv

load_dotenv()

from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_core.messages import (
    HumanMessage, SystemMessage, AIMessage, ToolMessage, BaseMessage,
)
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode
from langgraph.checkpoint.memory import MemorySaver
from pydantic import BaseModel, Field
import pandas as pd

llm = ChatOpenAI(model="gpt-4o-mini")


In [4]:
class SimpleState(TypedDict):
    message : str

def greet(state : SimpleState) -> dict:
    return {'message' : f'안녕, {state["message"]}'}

builder = StateGraph(SimpleState)
builder.add_node('greet', greet)
builder.add_edge(START, 'greet')
builder.add_edge('greet', END)

app = builder.compile()

In [5]:
app.invoke({'message' : 'abc'})

{'message': '안녕, abc'}

In [6]:
class TextState(TypedDict):
    text : str
    upper : str
    length : int

def to_upper(state:TextState) -> dict:
    return {'upper': state['text'].upper()}

def measure(state: TextState) -> dict:
    return {'length' : len(state['text'])}

builder = StateGraph(TextState)
builder.add_node('to_upper', to_upper)
builder.add_node('measure', measure)
builder.add_edge(START, 'to_upper')
builder.add_edge('to_upper', 'measure')
builder.add_edge('measure', END)
app = builder.compile()

In [7]:
app.invoke({'text' : 'hello langgraph', 'upper' : '', 'length' : 0})

{'text': 'hello langgraph', 'upper': 'HELLO LANGGRAPH', 'length': 15}

In [ ]:
# T5 -> finetuning 
# query -> T5 -> 검색 필요없는거
#                 simple 검색
#                multi-hop 검색

# chatgpt api, qwen
# query -> chatbot -> 검색 필요없는거 (direct 답변)
#                      rag 검색  
#                     tool 필요

In [28]:
# pydantic model
class QueryAnalysis(BaseModel):
    query : str = Field(description='원본쿼리')
    route: Literal["direct", "rag", "web_search"] = Field(
                    description = '라우팅 경로: direct(직접 답변), rag(문서 검색), web_search(웹 검색)')
    confidence : float = Field(description='분류 신뢰도(0.0~1.0)', ge=0.0, le=1.0) # greater than or equal
    reasoning : str = Field(description='분류 이유')


In [29]:
def analyze_query(query: str) -> QueryAnalysis:
    """LLM을 사용하여 쿼리를 분석하고 라우팅 경로를 결정"""
    system_prompt = """당신은 쿼리를 잘 분석하고 라우팅 경로를 분류하는 전문가입니다. 주어진 쿼리를 분석해서 최적의 처리 경로를 분석하세요
    
    라우팅 경로:
    direct : LLM 이 자체 지식으로 답변 가능한 일반 질문(상식, 개념 설명)
    rag : 회사 내부 문서나 특정 도메인 지식이 필요한 질문(정책, 매뉴얼, 사내 데이터)
    web_search : 최신 정보나 실시간 데이터가 필요한 질문 (뉴스, 시세, 날씨)
    
    예시:
    - 파이썬의 데코레이터란? -> direct (일반 프로그래밍 지식)
    - 우리 회사 연차 규정은? -> rag (내부 문서 필요)
    - 오늘 날씨는? -> web_search (실시간 정보 필요)
    """
    
    response = llm.with_structured_output(QueryAnalysis).invoke([
        SystemMessage(content = system_prompt), 
        HumanMessage(content=query)]) # direct , direct 쿼리입니다, ..
    
    return response
    

In [30]:
result = analyze_query("1+1은?")

/home/oncreative/anaconda3/envs/modu/lib/python3.11/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=QueryAnalysis(query='1+1...할 수 있습니다.'), input_type=QueryAnalysis])
  return self.__pydantic_serializer__.to_python(


In [31]:
result

QueryAnalysis(query='1+1은?', route='direct', confidence=1.0, reasoning='이 질문은 기본적인 산수 문제로, 수학적 계산이므로 LLM이 자체 지식으로 쉽게 답할 수 있습니다.')

In [32]:
# DetailedQueryAnalysis analyze_query_detailed
# route, confidence, sub_topic (str : 쿼리의 세부 주제), language (쿼리 언어, ko, en, other)

# analyze_query_detailed

In [33]:
# pydantic model
class DetailedQueryAnalysis(BaseModel):
    """상세 쿼리 분석 결과"""
    query : str = Field(description='원본쿼리')
    route: Literal["direct", "rag", "web_search"] = Field(
                    description = '라우팅 경로: direct(직접 답변), rag(문서 검색), web_search(웹 검색)')
    confidecnce : float = Field(description='분류 신뢰도(0.0~1.0)', ge=0.0, le=1.0) # greater than or equal
    reasoning : str = Field(description='분류 이유')
    sub_topic : str = Field(description='쿼리의 세부 주제')
    language : Literal["ko", "en", "other"] = Field(description = '쿼리 언어')

def analyze_query_detailed(query: str) -> DetailedQueryAnalysis:
    """LLM을 사용하여 쿼리를 분석하고 라우팅 경로를 결정"""
    system_prompt = """당신은 쿼리를 잘 분석하고 라우팅 경로를 분류하는 전문가입니다.
    주어진 쿼리를 분석해서 라우팅 경로, 세부 주제, 언어를 분류하세요
    
    라우팅 경로:
    direct : LLM 이 자체 지식으로 답변 가능한 일반 질문(상식, 개념 설명)
    rag : 회사 내부 문서나 특정 도메인 지식이 필요한 질문(정책, 매뉴얼, 사내 데이터)
    web_search : 최신 정보나 실시간 데이터가 필요한 질문 (뉴스, 시세, 날씨)
    
    언어 : ko(한국어), en(영어), other(기타)
    
    예시:
    - 파이썬의 데코레이터란? -> direct (일반 프로그래밍 지식)
    - 우리 회사 연차 규정은? -> rag (내부 문서 필요)
    - 오늘 날씨는? -> web_search (실시간 정보 필요)
    """
    
    response = llm.with_structured_output(QueryAnalysis).invoke([
        SystemMessage(content = system_prompt), 
        HumanMessage(content=query)]) # direct , direct 쿼리입니다, ..
    
    return response


In [34]:
# fallback -> 
from dataclasses import dataclass, field
from datetime import datetime

In [35]:
@dataclass
class RoutingConfig:
    """라우팅 전략을 설정"""
    confidence_threshold : float = 0.7
    fallback_route : str = 'rag'
    enable_logging : bool = True
    max_retires : int =2

@dataclass
class RoutingLog:
    """라우팅 로그"""
    timestamp : str
    query : str
    predicted_route : str
    actual_route : str
    confidence : float
    fallback_applied : bool


In [47]:
# AdvancedRoutingConfig -> route_weights (direct : 1.0, rag : 0.8, web_search : 0.6)
# weighted_route()  0.8
# confidence_threshold : 0.5 이하 -> fallback (rag)

@dataclass
class AdvancedRoutingConfig:
    confidence_threshold: float = 0.5
    fallback_route: str = "rag"
    route_weights: dict = field(default_factory=lambda: {
        "direct": 1.0, "rag": 0.8, "web_search": 0.6
    })

class WeightedRoutingEngine:
    def __init__(self, config: AdvancedRoutingConfig = None):
        self.config = config or AdvancedRoutingConfig()

    def weighted_route(self, analysis: QueryAnalysis) -> dict:
        weight = self.config.route_weights.get(analysis.route, 1.0)
        weighted_score = analysis.confidence * weight
        if weighted_score < self.config.confidence_threshold:
            final_route = self.config.fallback_route
            fallback = True
        else:
            final_route = analysis.route
            fallback = False
        return {
            "query": analysis.query[:25],
            "predicted": analysis.route,
            "confidence": analysis.confidence,
            "weight": weight,
            "weighted_score": round(weighted_score, 3),
            "final_route": final_route,
            "fallback": fallback
        }

In [48]:
weighted_engine = WeightedRoutingEngine()
test_q = '사내 보안 교육이 언제인가요?'
analysis = analyze_query(test_q)
result = weighted_engine.weighted_route(analysis)
print(result)

{'query': '사내 보안 교육이 언제인가요?', 'predicted': 'rag', 'confidence': 0.9, 'weight': 0.8, 'weighted_score': 0.72, 'final_route': 'rag', 'fallback': False}


/home/oncreative/anaconda3/envs/modu/lib/python3.11/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=QueryAnalysis(query='사...g'로 분류됩니다."), input_type=QueryAnalysis])
  return self.__pydantic_serializer__.to_python(


In [36]:
class RoutingEngine:
    def __init__(self, config: RountingConfig = None):
        self.config = config or RoutingConfig()
        self.logs : list[RoutingLog] = []
    
    def route(self, analysis: QueryAnalysis) -> str:
        """분석 결과를 바탕으로 최종 라우팅 경로를 결정"""
        predicted = analysis.route
        fallback_applied = False
        
        if analysis.confidence < self.config.confidence_threshold:
            actual = self.config.fallback_route
            fallback_applied = True
        else:
            actual = predicted
        
        if self.config.enable_logging:
            log = RoutingLog(timestamp = datetime.now().isoformat(), query = analysis.query,
                            predicted_route = predicted,
                            actual_route = actual,
                            confidence = analysis.confidence,
                            fallback_applied = fallback_applied)
            self.logs.append(log)
        
        return actual
    
    def get_stats(self) -> dict:
        if not self.logs:
            return {'total' : 0}
        total = len(self.logs)
        fabllbacks = sum(1 for log in self.logs if log.fallback_applied)
        route_dist = {}
        for log in self.logs:
            route_dist[log.actual_route] = route_dist.get(log.actual_route, 0) + 1
        avg_conf = sum(log.confidence for log in self.logs) / total
        return {
            'total' : total,
            'fallback_rate' : fallback/total,
            'avg_confidence' : avg_conf,
            'route_distribution' : route_dist
        }

In [37]:
engine = RoutingEngine(RoutingConfig(confidence_threshold=0.7))

In [38]:
analysis = analyze_query('파이썬 데코레이터가 뭔가요?')
final_route = engine.route(analysis)

/home/oncreative/anaconda3/envs/modu/lib/python3.11/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=QueryAnalysis(query='파...에 해당합니다.'), input_type=QueryAnalysis])
  return self.__pydantic_serializer__.to_python(


In [39]:
final_route

'direct'

In [40]:
engine.logs

[RoutingLog(timestamp='2026-05-12T21:04:38.898420', query='파이썬 데코레이터가 뭔가요?', predicted_route='direct', actual_route='direct', confidence=0.95, fallback_applied=False)]

In [41]:
from abc import ABC, abstractmethod # Abstract Base Class

In [42]:
class RouteHandler(ABC):
    @abstractmethod
    def handle(self, query:str) -> str:
        pass

In [43]:
class DirectHandler(RouteHandler):
    """LLM 이 직접 답변하는 경우"""
    def handle(self, query: str) -> str:
        response = llm.invoke(
                    [HumanMessage(content=query)])
        return response.content

class RAGHandler(RouteHandler):
    """RAG 검색해서 답변하는 경우"""
    def handle(self, query: str) -> str:
        return f"RAG '{query}'에 대한 문서 검색 결과 입니다"

class WebSearchHandler(RouteHandler):
    """웹 검색해서 답변하는 경우"""
    def handle(self, query: str) -> str:
        return f"WebSearch '{query}'에 대한 문서 검색 결과 입니다"

In [44]:
handlers = {
    'direct' : DirectHandler(),
    'rag' : RAGHandler(),
    'web_search' : WebSearchHandler()
}

In [46]:
test_q = '우리회사 복지정책에 대해서 알려줘'
analysis = analyze_query(test_q)
route = engine.route(analysis)
result = handlers[route].handle(test_q)
print(result)

RAG '우리회사 복지정책에 대해서 알려줘'에 대한 문서 검색 결과 입니다


/home/oncreative/anaconda3/envs/modu/lib/python3.11/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=QueryAnalysis(query='우...가 필요합니다.'), input_type=QueryAnalysis])
  return self.__pydantic_serializer__.to_python(


In [49]:
class AdaptiveRAGState(TypedDict):
    query : str
    route : str
    confidence : float
    reasoning : str
    response : str
    documents : list
    search_results : list

In [50]:
def query_analyzer_node(state : AdaptiveRAGState) -> dict:
    """쿼리 분석 노드"""
    query = state['query']
    analysis = analyze_query(query)
    return {'route' : analysis.route, 'confidence' : analysis.confidence, 'reasoning' : analysis.reasoning}

def direct_answer_node(state : AdaptiveRAGState) -> dict:
    """LLM 직접 답변 노드"""
    query = state['query']
    response = llm.invoke([HumanMessage(content=query)])
    return {'response' : f"[Direct] {response.content}"}

def rag_search_node(state : AdaptiveRAGState) -> dict:
    """RAG 검색 노드"""
    query = state['query']
    return {'documents' : [f"[Document] {query}에 대한 검색 문서"],
           'response' : f"[RAG] '{query}에 대한 문서 기반 답변'"}

def web_search_node(state : AdaptiveRAGState) -> dict:
    """웹 검색 노드"""
    query = state['query']
    return {'search_results' : [f"[Web] {query} 검색 결과"],
           'response' : f"[WebSearch] '{query}에 대한 웹 검색 답변'"}

In [51]:
def route_query(state: AdaptiveRAGState) -> dict:
    route = state['route']
    confidence = state['confidence']
    
    if confidence < 0.7:
        return 'rag_search'
    
    if route == 'direct':
        return 'direct_answer'
    elif route == 'rag':
        return 'rag_search'
    elif route == 'web_search':
        return 'web_search'
    else:
        return 'rag_search'

In [52]:
workflow = StateGraph(AdaptiveRAGState)
workflow.add_node('query_analyzer', query_analyzer_node)
workflow.add_node('direct_answer', direct_answer_node)
workflow.add_node('rag_search', rag_search_node)
workflow.add_node('web_search', web_search_node)

workflow.add_edge(START, 'query_analyzer')
workflow.add_conditional_edges(
    'query_analyzer',
    route_query,
    {
        'direct_answer' : 'direct_answer',
        'rag_search' : 'rag_search',
        'web_search' : 'web_search',
    }
)
workflow.add_edge('direct_answer', END)
workflow.add_edge('rag_search', END)
workflow.add_edge('web_search', END)

app = workflow.compile()

In [53]:
test_q = '우리 회사 재택근무 정책은?'
result = app.invoke({
    'query' : test_q,
    'route' : "",
    'confidence' : 0.0,
    'reasoning' : "",
    'response' : "",
    'documents' : [],
    'search_results' : []
})

/home/oncreative/anaconda3/envs/modu/lib/python3.11/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=QueryAnalysis(query='우...변이 가능하다.'), input_type=QueryAnalysis])
  return self.__pydantic_serializer__.to_python(


In [55]:
result

{'query': '우리 회사 재택근무 정책은?',
 'route': 'rag',
 'confidence': 0.9,
 'reasoning': '이 질문은 회사 내부의 재택근무 정책에 대한 정보가 필요하므로, 내부 문서를 참조해야 정확한 답변이 가능하다.',
 'response': "[RAG] '우리 회사 재택근무 정책은?에 대한 문서 기반 답변'",
 'documents': ['[Document] 우리 회사 재택근무 정책은?에 대한 검색 문서'],
 'search_results': []}

In [54]:
result['route']

'rag'

In [ ]:
def query_analyzer_node(state : AdaptiveRAGState) -> dict:
    """쿼리 분석 노드"""
    query = state['query']
    analysis = analyze_query(query)
    return {'route' : analysis.route, 'confidence' : analysis.confidence, 'reasoning' : analysis.reasoning}

def direct_answer_node(state : AdaptiveRAGState) -> dict:
    """LLM 직접 답변 노드"""
    query = state['query']
    response = llm.invoke([HumanMessage(content=query)])
    return {'response' : f"[Direct] {response.content}"}

def rag_search_node(state : AdaptiveRAGState) -> dict:
    """RAG 검색 노드"""
    query = state['query']
    return {'documents' : [f"[Document] {query}에 대한 검색 문서"],
           'response' : f"[RAG] '{query}에 대한 문서 기반 답변'"}

def web_search_node(state : AdaptiveRAGState) -> dict:
    """웹 검색 노드"""
    query = state['query']
    return {'search_results' : [f"[Web] {query} 검색 결과"],
           'response' : f"[WebSearch] '{query}에 대한 웹 검색 답변'"}

In [ ]:
# 1. fallback_node 추가 : confidence < 0.5 -> 사용자에게 재질문 요청, return 질문을 더 구체적으로 해주세요
# 2. route_query : 0.5보다 작으면 fallback
# 3. workflow

In [56]:
def fallback_node(state: AdaptiveRAGState) -> dict:
    """폴백 노드, 불확실한 쿼리가 입력되면 재질문 요청"""
    query = state['query'] # 이거 뭐야?
    response = llm.invoke([
        {'role' : 'user', 'content' : f"'{query}'라는 질문이 모호합니다. 더 구체적인 질문 3가지를 제안해주세요"}
    ])
    suggestions = response.content
    return {'response' : f"[Fallback] 질문을 더 구체적으로 해주세요.\n\n추천질문 : \n{suggestions}"}

def route_query_v2(state: AdaptiveRAGState) -> dict:
    route = state['route']
    confidence = state['confidence']
    
    if confidence < 0.5:
        return "fallback"
    
    if confidence < 0.7:
        return 'rag_search'
    
    if route == 'direct':
        return 'direct_answer'
    elif route == 'rag':
        return 'rag_search'
    elif route == 'web_search':
        return 'web_search'
    else:
        return 'rag_search'

In [57]:
workflow_v2 = StateGraph(AdaptiveRAGState)
workflow_v2.add_node('query_analyzer', query_analyzer_node)
workflow_v2.add_node('direct_answer', direct_answer_node)
workflow_v2.add_node('rag_search', rag_search_node)
workflow_v2.add_node('web_search', web_search_node)
workflow_v2.add_node('fallback', web_search_node)

workflow_v2.add_edge(START, 'query_analyzer')
workflow_v2.add_conditional_edges(
    'query_analyzer',
    route_query_v2,
    {
        'direct_answer' : 'direct_answer',
        'rag_search' : 'rag_search',
        'web_search' : 'web_search',
        'fallback' : 'fallback'
    }
)
workflow_v2.add_edge('direct_answer', END)
workflow_v2.add_edge('rag_search', END)
workflow_v2.add_edge('web_search', END)
workflow_v2.add_edge('fallback', END)

app = workflow_v2.compile()

In [60]:
test_q = '그거 뭐야?'
result = app.invoke({
    'query' : test_q,
    'route' : "",
    'confidence' : 0.0,
    'reasoning' : "",
    'response' : "",
    'documents' : [],
    'search_results' : []
})

/home/oncreative/anaconda3/envs/modu/lib/python3.11/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=QueryAnalysis(query='그...로 판단됩니다."), input_type=QueryAnalysis])
  return self.__pydantic_serializer__.to_python(


In [61]:
result

{'query': '그거 뭐야?',
 'route': 'direct',
 'confidence': 0.8,
 'reasoning': "질문의 내용이 구체적이지 않지만, '그거'라는 표현은 일반적인 지식이나 상식으로 설명할 수 있는 범위에 속할 가능성이 큽니다. 따라서 직접적인 대답이 가능할 것으로 판단됩니다.",
 'response': '[Direct] "그거 뭐야?"라는 질문은 상황에 따라 다르게 해석될 수 있어요. 궁금한 내용이나 주제를 좀 더 구체적으로 말씀해 주시면, 더 자세히 설명해 드릴 수 있습니다! 어떤 것을 궁금해 하시나요?',
 'documents': [],
 'search_results': []}